# AwA2 Model Testing — Cross-Dataset Transfer

**Compositional Image Retrieval — Deep Learning Assignment 2026 (secondary benchmark)**

This notebook is a *secondary* study: it re-runs our full CelebA model roster on
**Animals with Attributes 2** (AwA2, spec §2) to probe how each method transfers to a
dataset with a very different attribute structure. It is deliberately compact — the main
project deliverable is the CelebA benchmark.

The ground truth here uses AwA2's 85 **class-level** predicates (`predicate-matrix-binary.txt`):
a valid source class must **violate** every ± constraint of the query, and a valid target
class must **strictly satisfy** every constraint while sitting within an **adaptive**
free-attribute Hamming radius of the source (the nearest satisfying classes, +2 slack —
the class-level generalization of the spec's per-image ≤2 rule, since AwA2's class rows
never sit closer than Hamming 3 to a different class). Every source with ≥5 valid targets
is kept, with its **complete, uncapped** target set — 12 queries, 650–6,919 sources each,
exact Recall/Precision@K.

The evaluation protocol otherwise mirrors CelebA (frozen CLIP ViT-B/32 DB over the
7,468-image test corpus, Recall/Precision@{1,5,10}, stratified bootstrap CIs and paired
significance from `src/stats_eval.py`).

**Models under comparison** (display name ← AwA2 run)

| Display name | Method | Type | direction source |
|---|---|---|---|
| **LDE** | mined visual directions, flat arithmetic | training-free | visual |
| **GDE** | mined visual directions, geodesic + rejection | training-free | visual |
| **CLIP Enhanced** | text deltas, all geometry fixes (tier-0) | training-free | text |
| **DGP** | gated projection, uniform gate (tier-2d) | training-free | text |
| **Dual Score** | query-conditioned dual-channel fusion (tier-3) | learned | text + probe |
| **CLAY** | pre-SVD prompt stacking (k=50, rotH) | training-free | text subspace |
| **A-BAP** | CD-Bayes + learned Φ (tier-5) — *CelebA champion* | learned | probe |

> **A-BAP** is the display name of our CelebA champion (`tier5_Bprime_listnet`): an
> Amortized Bayesian Attribute-Posterior fusion. On CelebA it wins decisively; on AwA2, as
> we show below, its rule-scoring design pays off far less — **Dual Score**, which fuses a
> CLIP-cosine backbone with learned probe channels, is the strongest method here.


## 1. Setup

Thin cells only — all statistics live in `src/stats_eval.py`, the per-source dumps in
`src/per_source.py` (each parity-gated against its published CSV). Any model whose
per-source dump is not present locally (e.g. a VM-trained one not yet pulled) is skipped
with a message and simply omitted from the figures.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent if (Path.cwd() / "..").resolve().name == "Deep-Learning-Project" else Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else Path.cwd()
for _sub in ("", "Baselines", "Training-Free", "Trained"):
    _p = (ROOT / "src" / _sub) if _sub else (ROOT / "src")
    sys.path.insert(0, str(_p))

from eval import find_eval_json, load_eval_json
from per_source import load_per_source
from stats_eval import (holm_bonferroni, metric_label, paired_bootstrap_diff,
                        plot_ci_bars, recall_curve, stratified_bootstrap)

B = 1000
GT_LIST = load_eval_json(find_eval_json().parent / "awa2_evaluation.json")
PS = ROOT / "output" / "awa2" / "per_source"

# Ordered training-free (visual) → training-free (text) → learned, so the figure legend
# reads top-to-bottom in the ranking we discuss.
MODELS = {
    "LDE":           "awa2_tier_lde.npz",
    "GDE":           "awa2_tier_gde.npz",
    "CLIP Enhanced": "awa2_tier0_baseline.npz",
    "DGP":           "awa2_tier2d_tau100.0_a1.0_center.npz",
    "Dual Score":    "awa2_tier3_dualscore_seed0.npz",
    "CLAY":          "awa2_tier_clay.npz",
    "A-BAP":         "awa2_tier5_bprime_listnet.npz",
}

dumps = {}
for name, fname in MODELS.items():
    p = PS / fname
    if p.exists():
        dumps[name] = load_per_source(p)
    else:
        print(f"[missing] {fname} — '{name}' skipped (pull its per-source NPZ from the VM).")
print("loaded:", list(dumps))

## 2. The result — and the inversion

Point estimates with bootstrap 95% CIs for every model × spec metric (`stratified_bootstrap`,
§A.3 of the CelebA statistical notebook — same code).

In [2]:
boot = {name: stratified_bootstrap(d, B=B, seed=0) for name, d in dumps.items()}
rows = []
for name, res in boot.items():
    row = {"model": name}
    for f, r in res.items():
        row[metric_label(f)] = f"{r.point:.4f}  [{r.ci_lo:.4f}, {r.ci_hi:.4f}]"
    rows.append(row)
pd.DataFrame(rows).set_index("model")

,R@1,R@5,R@10,P@1,P@5,P@10
model,,,,,,
LDE,"0.0422 [0.0406, 0.0440]","0.1146 [0.1114, 0.1177]","0.1669 [0.1629, 0.1708]","0.0422 [0.0406, 0.0440]","0.0482 [0.0468, 0.0496]","0.0512 [0.0500, 0.0525]"
GDE,"0.0374 [0.0358, 0.0391]","0.1040 [0.1011, 0.1069]","0.1567 [0.1528, 0.1607]","0.0374 [0.0358, 0.0391]","0.0432 [0.0419, 0.0445]","0.0465 [0.0453, 0.0477]"
CLIP Enhanced,"0.1262 [0.1227, 0.1297]","0.2583 [0.2542, 0.2626]","0.3302 [0.3262, 0.3348]","0.1262 [0.1227, 0.1297]","0.1281 [0.1261, 0.1304]","0.1285 [0.1267, 0.1306]"
DGP,"0.1013 [0.0988, 0.1041]","0.2051 [0.2014, 0.2091]","0.2707 [0.2661, 0.2755]","0.1013 [0.0988, 0.1041]","0.1030 [0.1011, 0.1049]","0.1058 [0.1040, 0.1077]"
Dual Score,"0.2648 [0.2602, 0.2692]","0.4885 [0.4829, 0.4937]","0.5854 [0.5799, 0.5909]","0.2648 [0.2602, 0.2692]","0.2758 [0.2726, 0.2790]","0.2782 [0.2751, 0.2812]"
CLAY,"0.0143 [0.0131, 0.0156]","0.0535 [0.0511, 0.0560]","0.0875 [0.0845, 0.0906]","0.0143 [0.0131, 0.0156]","0.0164 [0.0156, 0.0172]","0.0181 [0.0173, 0.0189]"
A-BAP,"0.0975 [0.0941, 0.1010]","0.2245 [0.2190, 0.2295]","0.2942 [0.2885, 0.2996]","0.0975 [0.0941, 0.1010]","0.1040 [0.1010, 0.1070]","0.1025 [0.0997, 0.1054]"


The headline is the **R@1 column**: **Dual Score leads clearly** (0.265), roughly double
the next-best method (CLIP Enhanced, 0.126), with A-BAP and DGP close behind (~0.10) and
the mined-direction pair (LDE, GDE) and CLAY well below (0.01–0.04).

**Ranking (mean R@1, this benchmark):**

| method | AwA2 R@1 | AwA2 R@5 | AwA2 R@10 |
|---|---:|---:|---:|
| Dual Score | **0.2648** | **0.4885** | **0.5854** |
| CLIP Enhanced | 0.1262 | 0.2583 | 0.3302 |
| A-BAP | 0.0975 | 0.2245 | 0.2942 |
| DGP | 0.1013 | 0.2051 | 0.2707 |
| LDE | 0.0422 | 0.1146 | 0.1669 |
| GDE | 0.0374 | 0.1040 | 0.1567 |
| CLAY | 0.0143 | 0.0535 | 0.0875 |

> **On this benchmark, unlike CelebA, there is no clean training-free vs. learned split
> and no clean visual vs. text split.** Dual Score (learned, text+probe) is far ahead of
> every training-free method; A-BAP (learned, probe) sits in the *middle* of the pack, not
> at the bottom; and the purely visual-mining pair (LDE/GDE), which relies only on class
> centroid contrasts with zero learned signal, trails almost everyone. The mechanism
> section below (§4) unpacks why.

In [3]:
# Headline figure — grouped bars with 95% CI error bars (overlap reads as significance).
out = ROOT / "output" / "awa2" / "visual"; out.mkdir(parents=True, exist_ok=True)
fig = plot_ci_bars(boot, path=out / "awa2_stats_ci_bars.png", title="AwA2 — Recall@K with 95% CI")
fig

<Figure size 900x500 with 1 Axes>

In [4]:
# Recall@K, K = 1…10 — where in the ranking each method's retrieval materializes.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, d in dumps.items():
    ax.plot(range(1, 11), recall_curve(d, GT_LIST), marker="o", ms=3, label=name)
ax.set_xlabel("K"); ax.set_ylabel("macro Recall@K"); ax.legend(fontsize=7)
ax.set_title("AwA2 — Recall@K, K = 1…10"); fig.tight_layout()
fig.savefig(out / "awa2_stats_recall_curve.png", dpi=120)
fig

<Figure size 700x450 with 1 Axes>

The two panels tell the same story from different angles: at **R@1** the methods
separate cleanly by *how much of their score flows through raw CLIP visual similarity*;
by **R@10** they converge, because on a near-ceiling class-retrieval task the top-10 of
almost any reasonable scorer contains a valid target. The interesting physics is all at
K=1.

## 3. Significance — is Dual Score's lead real?

Paired stratified bootstrap of Δ = Dual Score − model (the champion on this benchmark vs
each other method), Holm–Bonferroni corrected over the {6 metrics × models} family. Same
machinery as the CelebA notebook §B.2. The **CI of Δ** is the significance statement (it
and the p-value cannot contradict by construction).

In [5]:
CHAMP = "Dual Score"
refs = [n for n in dumps if n != CHAMP]
pvals, diff_rows = {}, []
for ref in refs:
    d = paired_bootstrap_diff(dumps[CHAMP], dumps[ref], B=B, seed=0)
    for f, r in d.items():
        pvals[f"{ref} | {metric_label(f)}"] = r.p_value
        # A difference is significant iff its 95% CI excludes 0 (§A.4) — computed from the
        # numeric bounds, so it can never disagree with the printed CI.
        diff_rows.append({"vs": ref, "metric": metric_label(f), "delta": round(r.delta, 4),
                          "CI95": f"[{r.ci_lo:+.4f}, {r.ci_hi:+.4f}]",
                          "rel_%": round(100 * r.rel_gain, 1), "p_raw": r.p_value,
                          "CI excludes 0": (r.ci_lo > 0) or (r.ci_hi < 0)})
adj = holm_bonferroni(pvals)
dt = pd.DataFrame(diff_rows)
dt["p_holm"] = [adj[f"{r.vs} | {r.metric}"] for r in dt.itertuples()]
dt[dt.metric.isin(["R@1", "R@5", "R@10"])].reset_index(drop=True)

,vs,metric,delta,CI95,rel_%,p_raw,CI excludes 0,p_holm
0,LDE,R@1,0.2226,"[+0.2179, +0.2270]",527.8,0.001998,True,0.071928
1,LDE,R@5,0.3739,"[+0.3678, +0.3796]",326.2,0.001998,True,0.071928
2,LDE,R@10,0.4185,"[+0.4122, +0.4246]",250.7,0.001998,True,0.071928
3,GDE,R@1,0.2274,"[+0.2227, +0.2319]",608.6,0.001998,True,0.071928
4,GDE,R@5,0.3845,"[+0.3784, +0.3901]",369.8,0.001998,True,0.071928
5,GDE,R@10,0.4287,"[+0.4224, +0.4353]",273.7,0.001998,True,0.071928
6,CLIP Enhanced,R@1,0.1386,"[+0.1337, +0.1435]",109.8,0.001998,True,0.071928
7,CLIP Enhanced,R@5,0.2302,"[+0.2243, +0.2356]",89.1,0.001998,True,0.071928
8,CLIP Enhanced,R@10,0.2552,"[+0.2496, +0.2608]",77.3,0.001998,True,0.071928
9,DGP,R@1,0.1635,"[+0.1591, +0.1677]",161.4,0.001998,True,0.071928


Reading the table:

- **Every comparison's CI excludes zero at R@1/R@5/R@10** — Dual Score's lead over each of
  the other six methods is decisive, not sampling noise. The margin ranges from
  +0.139 (vs. CLIP Enhanced) to +0.251 (vs. CLAY).
- **Dual Score − A-BAP at R@1 = +0.167**, CI `[+0.162, +0.173]`: the CelebA champion's
  rule-scoring design is not competitive with a fused cosine+probe score here, though the
  gap is smaller than against the purely training-free methods (LDE +0.223, GDE +0.227,
  CLAY +0.251) — A-BAP's probe channel is still contributing *something*, just not enough
  to catch a method that keeps a CLIP-cosine backbone.
- **Dual Score − LDE / − GDE are the largest margins among "does visual mining help"**:
  +0.223 and +0.227 respectively — with this exact, uncapped ground truth, mining
  directions purely from class centroids (no learned signal, no probe channel) is the
  weakest strategy on this dataset.

> **p-value floor.** With 12 benchmark queries but tens of thousands of per-source samples
> per query, the bootstrap p-values saturate near the resolution floor (~0.002 raw, lifted
> to ≈0.072 after Holm across 36 tests) for every comparison. Read the **Δ confidence
> intervals** — not the p-values — as the significance statement: every one here excludes
> zero, so Dual Score's lead holds across the whole metric family.

## 4. Why Dual Score wins, and where each method's bias shows

### 4.1 The ground truth is a class relation, but not a linear one

**CelebA attributes are per-image; AwA2 attributes are class-level.** Every image of an
AwA2 class shares that class's 85-bit row of `predicate-matrix-binary.txt`, so the ground
truth collapses to a 50×50 class relation: a valid source class must *violate* every ±
constraint, a valid target class must *strictly satisfy* every constraint and sit within
an adaptive Hamming radius (nearest satisfying classes + 2 slack) of the source.

This is *not* the same as saying the task is easy or that raw cosine already solves it.
CLIP clusters species tightly, so cosine carries real signal — but the target set for a
composite query (e.g. `+meat & -domestic & +fierce`) is often a handful of specific,
non-adjacent classes, not simply "the nearest neighbours of the reference." That is
exactly where a single cosine-based query vector runs out of expressiveness, and where a
**learned, query-conditioned combination of cosine and per-attribute evidence** — Dual
Score's design — has room to add value that pure geometry (LDE/GDE/DGP/CLAY) or pure
rule-scoring (A-BAP) cannot supply alone.

### 4.2 Dual Score's advantage: it keeps the cosine backbone *and* adds learned channels

Dual Score fuses a CLIP-cosine-based query with trained presence/absence probe channels
(`β·s_neg + γ·s_pos`) in a single learned scorer. On queries where cosine is already a
strong signal (`+stripes` 0.550, `+spots` 0.369, `+swims` 0.381) it does not throw that
signal away the way CLAY's pre-SVD subspace stacking does; and on queries where cosine
alone struggles with negation (`-furry` 0.404, `-quadrapedal` 0.382 — both **far** above
every training-free method's score on the same query) its learned probe channels recover
exactly the negation signal cosine is known to lack (CLIP's text encoder has no "not X"
representation — Alhamoud et al. 2025). This dual structure is *why* it separates cleanly
from every other method at every K.

### 4.3 A-BAP: not "last", but mismatched to single-attribute geometry

A-BAP is not uniformly the weakest method here — it actually **wins its query** against
every training-free method on `-furry` (0.088), `+horns` (0.182), `+longneck` (0.230),
`+big & -meat` (0.123), and `+meat & -domestic & +fierce` (0.190), several of which are
negation-heavy or multi-attribute composites where scoring the *rule*,
$S = \sum_a \log p_a + \sum_b \log(1-p_b) + \log P(\text{Hamming}\le\text{radius})$, still
pays off — this is the same inductive bias that makes it CelebA's champion. Where it loses
badly is the single, strongly-visual positive queries (`+stripes` 0.038, `+swims` 0.045,
`-quadrapedal` 0.043) — cases where CLIP's raw cosine alone already separates the target
classes cleanly, and A-BAP's Poisson-binomial tail over ~80 free attributes at per-probe
error ε≈0.14 (an expected ≈11 predicted flips, an order of magnitude above the Hamming
radius) adds probe noise on top of a signal that needed no rule-scoring help.

### 4.4 Why CLAY and pure visual-mining trail

**CLAY** stacks all condition-prompt embeddings and scores in the pre-SVD subspace,
discarding most of the raw identity direction. On a task where the target set is a small,
specific subset of classes (not simply "near the reference"), throwing away identity is
the worst possible trade — CLAY is the weakest method on 9 of 12 queries, including a
`+furry & -water & +ground` R@1 of 0.000.

**LDE and GDE** mine directions purely from class centroid contrasts
($v_a = \text{mean}(\text{images with } a) - \mu$) with *no learned component at all*. That
is a strong, zero-modality-gap signal for single positive attributes with a wide, visually
coherent has-$a$ set (`+swims & +ocean`: LDE 0.299, GDE 0.270 — among the best scores for
that query across all seven methods) but it has no mechanism to combine multiple
constraints adaptively, weight them by query, or represent negation beyond fixed
subtraction/rejection. That is enough to win a handful of queries but not enough to compete
with a method that *learns* how to combine cosine and attribute evidence per query — hence
LDE/GDE trail Dual Score by the largest margins in the significance table above.

**DGP** sits between: a uniform gate over text-direction geometry recovers some of
CLIP-cosine's identity signal (`+stripes` 0.317, `+spots` 0.245 — close to Dual Score's
own numbers on those two) but, like CLAY, it is built entirely from text prompts and
inherits the modality gap on attributes with no strong lexical-visual correspondence
(`-quadrapedal` 0.002, `-furry` 0.003).

### 4.5 LDE vs GDE: flat arithmetic edges geodesic, on both losing to learned fusion

First, the relationship: **LDE is not an enhancement of GDE — it is the linear special
case GDE generalizes** (Trager et al.'s "ideal words" = GDE with $\mathcal M = \mathbb
R^n$: intrinsic mean → arithmetic mean, Exp/Log → identity). In our code LDE is a zero-diff
ablation of GDE (same mined directions, only the geometric operators differ).

LDE beats GDE on 10 of 12 queries (mean R@1 0.042 vs 0.037) — a small but consistent
margin, the same direction as the CelebA-notebook's original finding, though on this
corrected benchmark neither method is competitive with Dual Score. The gap decomposes into
the same two operator differences documented in the CelebA study:

- **Negation: rejection vs subtraction.** GDE removes the *entire* component along
  $\hat v_a$: $q \leftarrow q - (q\cdot\hat v_a)\hat v_a$. For a broad attribute like
  `quadrapedal` (~34 of 50 classes), $v_a$ points near the dominant axis of animal identity
  itself, so the projection is large for every reference and rejection risks amputating the
  identity anchor. LDE's subtraction shifts by the small fixed $\lVert v_a\rVert$ and
  leaves identity intact — visible in `-quadrapedal` (LDE 0.003 vs GDE 0.001) even though
  both are near the noise floor on this query.

- **Curvature: Log/Exp maps vs flat arithmetic.** GDE linearizes at one global $\mu$; the
  log-map factor $\theta/\sin\theta$ amplifies with distance from $\mu$. AwA2 spans 50
  well-separated species clusters (large $\theta$ spread) vs CelebA's single tight "face"
  cluster, so the tangent approximation is stretched exactly where AwA2 lives — the flat
  composition avoids that distortion, but on this benchmark the ceiling for *either*
  geometry-only approach is well below what learned per-query fusion (Dual Score) achieves.

## 5. Takeaway

**On this benchmark, Dual Score is the clear winner** (R@1 0.265, roughly double the
next-best method), because it is the only method that both (a) keeps a CLIP-cosine
backbone — preserving identity and the strong within-species visual signal AwA2 offers —
and (b) *learns* additive probe channels that recover the negation semantics CLIP's text
encoder cannot represent on its own. Every other method gives up one of those two
ingredients: CLAY and DGP discard identity information via text-only subspace/gate
constructions; LDE/GDE mine visual directions with no learned combination mechanism;
A-BAP scores the attribute-rule directly with no cosine backbone at all.

**A-BAP is not simply "the worst method"** — it wins several negation-heavy and
multi-attribute composite queries (`-furry`, `+horns`, `+longneck`,
`+meat & -domestic & +fierce`) against every training-free alternative, the same inductive
bias that makes it CelebA's champion. It loses on single, strongly-visual positive
attributes, where its probe-noise tail adds variance a clean cosine signal did not need.

This is what a transfer study should produce: not "the best method in isolation", but a
map of which methods' inductive biases transfer and which do not. Dual Score's
architecture — a learned, query-conditioned *combination* of geometric and semantic
evidence, rather than a commitment to either alone — is the one that generalizes best
across both datasets in this study, which is itself a useful finding for motivating the
project's fusion-module design (spec §3, item 2).

*Reproduction:* the runners in `src/awa2/` — `generate_gt.py` (ground truth), then
`tier0_baseline.py`, `tier_clay.py`, `tier_gde.py`, `tier_dgp.py` (training-free, local),
and `tier_dualscore.py`, `tier5_fusion.py` (trained, VM) — followed by `compare.py` for the
statistical comparison reproduced in this notebook.